In [71]:
from pathlib import Path
import json
import subprocess
import numpy as np
import pandas as pd

In [72]:
from setup.structure import parameters_from_tar_filename

In [89]:
DATA_ROOT = Path("./data/res")
APKDIFF = DATA_ROOT / "apkdiff.json"
DEX_SORT = DATA_ROOT / "dex_sort.json"
TARS_ROOT = Path("./data/build/tars")
CLASSES_ROOT = Path("./data/build/classes")

In [74]:
def show_full_df(df):
    with pd.option_context(
        "display.max_rows", None, "display.max_columns", None
    ):  # more options can be specified also
        display(df)


In [75]:
# Load the JSON file into a Python dictionary (replace 'your_file.json' with your actual file path)
with open(DEX_SORT, "r") as file:
    data = json.load(file)

# Flatten the data into a format we can use for DataFrame creation
flattened_data = []

# Iterate through each entry
for filename, file_data in data.items():
    # Process the 'playstore' and 'local' dictionaries
    for category in ["playstore", "local"]:
        for hash_key, dex_file in file_data.get(category, {}).items():
            flattened_data.append(
                {
                    "filename": filename,
                    "category": category,
                    "hash_key": hash_key,
                    "dex_file": dex_file,
                }
            )

    # Process the 'differing_dexes' dictionary
    for hash_key, differing_info in file_data.get("differing_dexes", {}).items():
        flattened_data.append(
            {
                "filename": filename,
                "category": "differing_dexes",
                "hash_key": hash_key,
                "dex_file": differing_info,
            }
        )

# Convert the flattened data into a pandas DataFrame
df = pd.DataFrame(flattened_data)

# Set 'filename', 'category', and 'hash_key' as the index
df.set_index(["filename", "category", "hash_key"], inplace=True)
df

dex_file
filename                                        category        hash_key                                                                   
signal-android_v7.30.2.tar.gz                   playstore       a4decdb3a060d81ecc3f59f8ee02e625629ed2b17546d96...             classes3.dex
                                                                c061b0748ea8654fa3a585f4e11557e35889517eecb732f...             classes4.dex
                                                                5b2bd0e9dce85b78981212f3b41cd55e87d0c3c0017911a...              classes.dex
                                                                98059dae55a3c4c379e9a32f1b38b8a347bd6fb39ed87b0...             classes7.dex
                                                                f32782128a6145b054f1a5c65cb05902ed002e71033ce1f...             classes2.dex
...                                                                                                                                     ...
signal-android-ctime-reversed_v7.28.4_03.tar.gz local           68e80cf5bbbe30b734c1806bea25cb5fddab86ff3a59ead...              classes.dex
                                                                1ea7370e3a24cfe9e9c2a84853546a54d6faa0a95c0f7c3...             classes2.dex
                                                                c0f14f1f46bc39a9f6d64651f102089b67ce61090dc4a5b...             classes5.dex
                                                                9b27d12a5cef9990c2d0b674489c476ff490749dd109d7c...             classes6.dex
                                                differing_dexes f5ea9a4a2748e34879409b7fb1184ad4d86a1e755f5f92c...  playstore->classes7.dex

[398 rows x 1 columns]

In [77]:
def find_differing_file_names(df: pd.DataFrame, sort_keys=True):
    # Extract rows for playstore and local categories
    playstore_df = df[df.index.get_level_values("category") == "playstore"]
    local_df = df[df.index.get_level_values("category") == "local"]

    # Merge the playstore and local dataframes on the hash_key
    merged_df = playstore_df.merge(
        local_df, on="hash_key", suffixes=("_playstore", "_local"), how="inner"
    )

    # Find rows where dex_file names are different between playstore and local
    differing_files = merged_df[
        merged_df["dex_file_playstore"] != merged_df["dex_file_local"]
        ]

    # Return the differing files with relevant info
    if sort_keys:
        return differing_files[["dex_file_playstore", "dex_file_local"]].sort_values(
            by=["dex_file_playstore"]
        )
    else:
        return differing_files[["dex_file_playstore", "dex_file_local"]]


In [78]:
# Initialize an empty list to store the results
all_differing_files = []

# Iterate over each filename in the DataFrame and find differing files
for filename in df.index.get_level_values("filename").unique():
    differing_files_df = find_differing_file_names(df.loc[filename])
    differing_files_df["filename"] = filename  # Add filename column
    all_differing_files.append(differing_files_df)

In [79]:
dex_mapping_df = pd.concat(all_differing_files)
dex_mapping_df = dex_mapping_df.reset_index().set_index(["filename", "hash_key"])
all_filenames = set(df.index.get_level_values("filename").tolist())
to_remove_filenames = set(dex_mapping_df.index.get_level_values("filename").tolist())

In [80]:
wierdos_fnames = list(all_filenames - to_remove_filenames)
wierdos_fnames # Files where none of the dex hashes match :)

['signal-android-ctime-sort_v7.28.4.tar.gz',
 'signal-android-ctime-reversed_v7.28.4_01.tar.gz',
 'dfstest-signal-android-ctime-reversed_v7.28.4_02.tar.gz',
 'dfstest-signal-android-ctime-reversed_v7.28.4_01.tar.gz']

In [81]:
wierdos_fnames[:1]

['signal-android-ctime-sort_v7.28.4.tar.gz']

In [82]:
def is_lfs_pointer(path):
    try:
        if not path.is_file():
            return False
        if path.stat().st_size > 1024:  # usually real files are larger than pointer files
            return False
        with open(path, "r") as f:
            first_line = f.readline()
            return first_line.startswith("version https://git-lfs.github.com/spec/")
    except Exception:
        return False

In [83]:
for fname in wierdos_fnames:
    class_path = TARS_ROOT / "../classes" / fname
    class_path.mkdir(exist_ok=True, parents=True)

In [87]:
# for tar_name in wierdos_fnames:
#     tar_path = TARS_ROOT / tar_name
#     if not tar_path.exists() or is_lfs_pointer(tar_path):
#         print(f"Pulling actual content for: {tar_name}")
#         try:
#             cmd = ["git", "lfs", "pull", "--include", str(tar_path)]
#             subprocess.run(cmd, check=True)
#         except subprocess.CalledProcessError as e:
#             print(f"Failed to pull {tar_name}: {e}")
#     else:
#         print(f"File ready: {tar_name}")

print("Manually extract the files into ./data/build/classes/")

Manually extract the files into ./data/build/classes/


In [85]:
from androguard.core import dex

In [88]:
TARS_ROOT

PosixPath('data/build/tars')

Androguard version: 4.1.2


ModuleNotFoundError: No module named 'androguard.core.bytecodes'